# 01 — Data Understanding

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Objective
Understand exactly what the data is before any cleaning, EDA, or modeling.
Load CARE-to-Compare **Wind Farm A** (5 onshore Portuguese turbines from the EDP
open dataset) and answer:
- How many datasets, and how are they organised (normal vs event runs)?
- What are the columns and column groups?
- Sampling interval and time span?
- How many turbines, and how are they identified?
- What does `status_type_id` mean, and how imbalanced is it?
- What real fault events exist, on which turbines, for which components?

**No cleaning / feature engineering / modeling here — just understanding.**

## Key known constraint
Only ~12 real fault events across 5 turbines. This scarcity drives the whole
project: event-based validation (not random splits), anomaly detection as a
co-lead, and PR-AUC/recall over accuracy.

In [1]:
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
DATASETS_DIR = BASE / "datasets"

print("Base path:", BASE.resolve())
print("Exists:", BASE.exists())
print("Datasets dir exists:", DATASETS_DIR.exists())

Base path: /Users/aayushaswal/wind-turbine-predictive-maintenance/data/raw/Wind Farm A
Exists: True
Datasets dir exists: True


In [2]:
csv_files = sorted(DATASETS_DIR.glob("*.csv"), key=lambda p: int(p.stem))
print(f"Number of dataset CSVs: {len(csv_files)}")
print("Dataset IDs:", [p.stem for p in csv_files])

for p in csv_files:
    print(f"  {p.name:10s} {p.stat().st_size/1e6:7.1f} MB")

Number of dataset CSVs: 22
Dataset IDs: ['0', '3', '10', '13', '14', '17', '22', '24', '25', '26', '38', '40', '42', '45', '51', '68', '69', '71', '72', '73', '84', '92']
  0.csv         36.9 MB
  3.csv         37.1 MB
  10.csv        35.8 MB
  13.csv        36.2 MB
  14.csv        36.6 MB
  17.csv        36.9 MB
  22.csv        35.5 MB
  24.csv        37.0 MB
  25.csv        36.6 MB
  26.csv        36.1 MB
  38.csv        37.3 MB
  40.csv        37.3 MB
  42.csv        35.9 MB
  45.csv        36.5 MB
  51.csv        36.5 MB
  68.csv        36.4 MB
  69.csv        36.7 MB
  71.csv        36.5 MB
  72.csv        36.2 MB
  73.csv        36.3 MB
  84.csv        36.5 MB
  92.csv        36.1 MB


## 1. Fault events — the ground truth

`event_info.csv` lists each of the 22 datasets as either an **anomaly** run
(leads to a real fault) or a **normal** run. This is the closest thing to the
original EDP failure logbook and defines what we ultimately predict.

In [3]:
events = pd.read_csv(BASE / "event_info.csv", sep=";")

print("Event table shape:", events.shape)
print("\nColumns:", list(events.columns))
print("\nLabel breakdown:")
print(events["event_label"].value_counts())

print("\nEvents per turbine (asset):")
print(events["asset"].value_counts().sort_index())

print("\nFault types (anomaly events only):")
anomalies = events[events["event_label"] == "anomaly"]
print(anomalies["event_description"].value_counts())

# full table for reference
events.sort_values("asset")

Event table shape: (22, 8)

Columns: ['asset', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description']

Label breakdown:
event_label
anomaly    12
normal     10
Name: count, dtype: int64

Events per turbine (asset):
asset
0     5
10    5
11    4
13    4
21    4
Name: count, dtype: int64

Fault types (anomaly events only):
event_description
Hydraulic group              6
Gearbox failure              2
Generator bearing failure    2
Transformer failure          1
Gearbox bearings damaged     1
Name: count, dtype: int64


,asset,event_id,event_label,event_start,event_start_id,event_end,event_end_id,event_description
18,0,71,normal,2023-01-02 00:00:00,52439,2023-01-16 00:00:00,54455,NaN
3,0,73,anomaly,2023-06-10 11:40:00,52745,2023-06-17 11:40:00,53753,Hydraulic group
4,0,0,anomaly,2023-08-06 06:10:00,52436,2023-08-20 06:10:00,54447,Generator bearing failure
5,0,26,anomaly,2023-10-12 10:20:00,52261,2023-10-19 10:20:00,53269,Hydraulic group
14,0,24,normal,2023-04-27 15:00:00,52720,2023-05-11 11:20:00,54714,NaN
6,10,40,anomaly,2022-12-26 00:00:00,51363,2023-01-26 13:00:00,55870,Generator bearing failure
7,10,42,anomaly,2023-09-09 15:50:00,52303,2023-09-16 15:50:00,53309,Hydraulic group
8,10,10,anomaly,2023-10-11 08:40:00,52611,2023-10-18 08:40:00,53591,Gearbox failure
16,10,17,normal,2023-11-02 15:20:00,52597,2023-11-16 00:40:00,54513,NaN
15,10,3,normal,2023-04-27 03:00:00,52185,2023-05-18 01:10:00,55198,NaN


## 2. Inspect one anomaly dataset

Load one anomaly run (dataset 0 — asset 0, generator bearing failure) and
confirm the label structure: `status_type_id` should contain `0` (normal) and
`4` (pre-failure degradation). Measure how small the positive class is.

In [4]:
df0 = pd.read_csv(DATASETS_DIR / "0.csv", sep=";")

print("Dataset 0 shape:", df0.shape)
print("Asset(s) in this file:", df0["asset_id"].unique())
print("Time span:", df0["time_stamp"].min(), "->", df0["time_stamp"].max())

print("\ntrain_test column values:")
print(df0["train_test"].value_counts())

print("\nstatus_type_id value counts:")
print(df0["status_type_id"].value_counts().sort_index())

print("\nstatus_type_id as % of rows:")
print((df0["status_type_id"].value_counts(normalize=True).sort_index() * 100).round(3))

Dataset 0 shape: (54986, 86)
Asset(s) in this file: [0]
Time span: 2022-08-04 06:10:00 -> 2023-08-24 06:10:00

train_test column values:
train_test
train         52148
prediction     2838
Name: count, dtype: int64

status_type_id value counts:
status_type_id
0    47540
3     1207
4     6239
Name: count, dtype: int64

status_type_id as % of rows:
status_type_id
0    86.458
3     2.195
4    11.347
Name: proportion, dtype: float64


## 3. Farm-wide label & structure scan

Dataset 0 revealed a third label — `status_type_id = 3` — not just 0/4, and a
`train_test` column with values `train`/`prediction`. Before designing the
target, scan all 22 datasets to answer:
- Which `status_type_id` values appear, and how often, across the whole farm?
- Does each file contain exactly one asset?
- Do anomaly files contain status 4 and normal files contain none?
- What's the real farm-wide class balance (status 4 vs everything)?

In [5]:
records = []
status_all = {}

for p in csv_files:
    d = pd.read_csv(p, sep=";", usecols=["time_stamp", "asset_id", "train_test", "status_type_id"])
    ev_id = int(p.stem)
    label_row = events.loc[events["event_id"] == ev_id]
    ev_label = label_row["event_label"].iloc[0] if len(label_row) else "??"
    ev_desc = label_row["event_description"].iloc[0] if len(label_row) else "??"

    vc = d["status_type_id"].value_counts()
    for k, v in vc.items():
        status_all[k] = status_all.get(k, 0) + v

    records.append({
        "event_id": ev_id,
        "event_label": ev_label,
        "description": ev_desc,
        "asset": d["asset_id"].unique().tolist(),
        "rows": len(d),
        "n_status4": int((d["status_type_id"] == 4).sum()),
        "n_status3": int((d["status_type_id"] == 3).sum()),
        "n_status0": int((d["status_type_id"] == 0).sum()),
    })

scan = pd.DataFrame(records).sort_values(["event_label", "event_id"])
print("Farm-wide status_type_id totals:")
for k in sorted(status_all):
    print(f"  status {k}: {status_all[k]:>8,} rows")

total_rows = sum(status_all.values())
print(f"\nTotal rows across farm: {total_rows:,}")
print(f"status 4 as % of all rows: {100*status_all.get(4,0)/total_rows:.2f}%")

scan

Farm-wide status_type_id totals:
  status 0: 1,053,736 rows
  status 3:   23,689 rows
  status 4:  119,322 rows

Total rows across farm: 1,196,747
status 4 as % of all rows: 9.97%


,event_id,event_label,description,asset,rows,n_status4,n_status3,n_status0
0,0,anomaly,Generator bearing failure,[0],54986,6239,1207,47540
2,10,anomaly,Gearbox failure,[10],53592,7129,1292,45171
6,22,anomaly,Hydraulic group,[21],53036,7910,1323,43803
9,26,anomaly,Hydraulic group,[0],53702,6239,1258,46205
11,40,anomaly,Generator bearing failure,[10],56158,8827,1584,45747
12,42,anomaly,Hydraulic group,[10],53886,6850,1291,45745
13,45,anomaly,Hydraulic group,[13],53739,3997,433,49309
14,51,anomaly,Gearbox bearings damaged,[21],54436,6044,1434,46958
15,68,anomaly,Transformer failure,[11],54358,2014,281,52063
18,72,anomaly,Gearbox failure,[21],54082,6044,1196,46842


In [6]:
# Verify: is status_type_id an operating code (present everywhere) rather than a fault label?
print("status_type_id present in NORMAL files (should be non-zero if it's an operating code):")
normal_ids = events.loc[events["event_label"] == "normal", "event_id"].tolist()
anomaly_ids = events.loc[events["event_label"] == "anomaly", "event_id"].tolist()

print(f"  normal files with any status-4 rows: "
      f"{sum(scan.loc[scan['event_label']=='normal','n_status4'] > 0)} / {len(normal_ids)}")
print(f"  anomaly files with any status-4 rows: "
      f"{sum(scan.loc[scan['event_label']=='anomaly','n_status4'] > 0)} / {len(anomaly_ids)}")

# Now inspect the actual event window in one anomaly file, using event_start/event_end
ev = events.loc[events["event_id"] == 0].iloc[0]
print(f"\nEvent 0: asset {ev['asset']}, '{ev['event_description']}'")
print(f"  event_start: {ev['event_start']}  (id {ev['event_start_id']})")
print(f"  event_end:   {ev['event_end']}  (id {ev['event_end_id']})")

d0 = pd.read_csv(DATASETS_DIR / "0.csv", sep=";",
                 usecols=["time_stamp", "id", "status_type_id"])
print(f"  file id range: {d0['id'].min()} -> {d0['id'].max()}")
print(f"  file time range: {d0['time_stamp'].min()} -> {d0['time_stamp'].max()}")

# Does the event window fall inside this file's id range?
in_window = d0[(d0["id"] >= ev["event_start_id"]) & (d0["id"] <= ev["event_end_id"])]
print(f"  rows inside event_start_id..event_end_id: {len(in_window)}")
print(f"  status breakdown INSIDE the event window:")
print(in_window["status_type_id"].value_counts().sort_index())

status_type_id present in NORMAL files (should be non-zero if it's an operating code):
  normal files with any status-4 rows: 10 / 10
  anomaly files with any status-4 rows: 12 / 12

Event 0: asset 0, 'Generator bearing failure'
  event_start: 2023-08-06 06:10:00  (id 52436)
  event_end:   2023-08-20 06:10:00  (id 54447)
  file id range: 0 -> 54985
  file time range: 2022-08-04 06:10:00 -> 2023-08-24 06:10:00
  rows inside event_start_id..event_end_id: 2012
  status breakdown INSIDE the event window:
status_type_id
4    2012
Name: count, dtype: int64


## 4. Data Understanding — Summary

### Dataset
- **Source:** CARE-to-Compare Wind Farm A (5 onshore Portugal turbines, from EDP open data).
- **22 datasets** (~36 MB / ~54k rows each): **12 anomaly runs, 10 normal runs**.
- **5 turbines:** assets 0, 10, 11, 13, 21.
- **Sampling:** 10-minute intervals. Timestamps anonymised (start ~2022);
  intervals and ordering are real, absolute dates are not.
- **86 columns:** `time_stamp`, `asset_id`, `id`, `train_test`, `status_type_id`,
  + 81 sensor columns (mostly `_avg`; a few with `_min`/`_max`/`_std`; 8 bare counters).

### Critical finding — the real label is NOT status_type_id
- `status_type_id` (values 0, 3, 4) is a **SCADA operating-state code present in
  ALL turbines**, including healthy ones — every normal file also contains
  status-4 rows. It is a **feature / filter, not the fault label**.
- The true label lives in **`event_info.csv`**: `event_label` (anomaly/normal)
  marks which runs lead to a fault, and `event_start_id`/`event_end_id` define
  the fault window via the **`id`** column (verified: exact, clean mapping).

### Fault types (12 anomalies)
- Hydraulic group: 6 · Gearbox / gearbox bearings: 3 · Generator bearing: 2 · Transformer: 1.
- → Only a **binary** target ("pre-failure period vs normal") is defensible.
  Per-component prediction is impossible (1 transformer, etc.) — stated as a limitation.

### Decisions this sets for later notebooks
1. **Target (NB05):** build from `event_start_id`/`event_end_id` on the `id` column,
   not from `status_type_id`. Consider a lead-time window before `event_start`.
2. **Validation (NB05):** only 22 runs / 12 events / 5 turbines → **leave-turbines-out**
   (or leave-events-out). Random row splits are invalid and would leak massively.
3. **status 3 (NB03):** curtailed/derated operating rows — decide to keep as feature
   or filter out; do not treat as fault.
4. **Evaluation (NB06):** event-level scarcity dominates → PR-AUC + recall, never accuracy.

### Next: `02_eda.ipynb`
Sensor distributions and — most important — how sensor behaviour (bearing/gearbox/
transformer temps) changes approaching the fault window vs normal operation.